In [3]:
!pip install transformers datasets seqeval
!pip install -U datasets huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=106f0a1ecf95a720dcb94dd2a3eb82fe93fecea2438539b46321ae2399c876e3
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 137.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.4.2
    Uninstalling hf-xet-1.4.2:
      S

In [4]:
!wget https://data.deepai.org/conll2003.zip
!unzip conll2003.zip

--2026-04-06 15:15:43--  https://data.deepai.org/conll2003.zip
Resolving data.deepai.org (data.deepai.org)... 84.17.38.231, 2400:52e0:1500::1096:1
Connecting to data.deepai.org (data.deepai.org)|84.17.38.231|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 982975 (960K) [application/zip]
Saving to: ‘conll2003.zip’

conll2003.zip       100%[===================>] 959.94K  5.57MB/s    in 0.2s    

2026-04-06 15:15:44 (5.57 MB/s) - ‘conll2003.zip’ saved [982975/982975]

Archive:  conll2003.zip
  inflating: metadata                
  inflating: test.txt                
  inflating: train.txt               
  inflating: valid.txt               


In [5]:
!wget https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-train.conllu
!wget https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-dev.conllu

--2026-04-06 15:15:46--  https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-train.conllu
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15029817 (14M) [text/plain]
Saving to: ‘en_ewt-ud-train.conllu’

en_ewt-ud-train.con 100%[===================>]  14.33M  --.-KB/s    in 0.08s   

2026-04-06 15:15:47 (170 MB/s) - ‘en_ewt-ud-train.conllu’ saved [15029817/15029817]

--2026-04-06 15:15:48--  https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-dev.conllu
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request s

In [6]:
def read_conll(file_path):
    sentences, labels = [], []
    words, tags = [], []

    with open(file_path, "r") as f:
        for line in f:
            if line.strip() == "":
                if words:
                    sentences.append(words)
                    labels.append(tags)
                    words, tags = [], []
            else:
                splits = line.strip().split()
                words.append(splits[0])
                tags.append(splits[-1])  # chunk tag

    return sentences, labels

train_sents_chunk, train_labels_chunk = read_conll("train.txt")
val_sents_chunk, val_labels_chunk = read_conll("valid.txt")

In [7]:
def read_conllu(file_path):
    sentences, labels = [], []
    words, tags = [], []

    with open(file_path, "r") as f:
        for line in f:
            if line.startswith("#"):
                continue
            if line.strip() == "":
                if words:
                    sentences.append(words)
                    labels.append(tags)
                    words, tags = [], []
            else:
                parts = line.strip().split("\t")
                if "-" in parts[0] or "." in parts[0]:
                    continue
                words.append(parts[1])
                tags.append(parts[3])  # UPOS tag

    return sentences, labels

train_sents_pos, train_labels_pos = read_conllu("en_ewt-ud-train.conllu")
val_sents_pos, val_labels_pos = read_conllu("en_ewt-ud-dev.conllu")

In [8]:
def create_label_map(labels):
    unique_labels = sorted(list(set(tag for sent in labels for tag in sent)))
    label2id = {l: i for i, l in enumerate(unique_labels)}
    id2label = {i: l for l, i in label2id.items()}
    return unique_labels, label2id, id2label

chunk_labels, chunk_l2i, chunk_i2l = create_label_map(train_labels_chunk)
pos_labels, pos_l2i, pos_i2l = create_label_map(train_labels_pos)

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_and_align(sentences, labels, label2id):
    encodings = tokenizer(
        sentences,
        truncation=True,
        padding=True,
        is_split_into_words=True
    )

    aligned_labels = []

    for i, label in enumerate(labels):
        word_ids = encodings.word_ids(batch_index=i)
        prev = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != prev:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)
            prev = word_idx

        aligned_labels.append(label_ids)

    encodings["labels"] = aligned_labels
    return encodings

# Apply
train_chunk_enc = tokenize_and_align(train_sents_chunk, train_labels_chunk, chunk_l2i)
val_chunk_enc   = tokenize_and_align(val_sents_chunk, val_labels_chunk, chunk_l2i)

train_pos_enc = tokenize_and_align(train_sents_pos, train_labels_pos, pos_l2i)
val_pos_enc   = tokenize_and_align(val_sents_pos, val_labels_pos, pos_l2i)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
import torch

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])

In [11]:
from transformers import AutoModelForTokenClassification

model_chunk = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(chunk_labels),
    id2label=chunk_i2l,
    label2id=chunk_l2i
)

model_pos = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(pos_labels),
    id2label=pos_i2l,
    label2id=pos_l2i
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [12]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    logging_steps=50
)

In [13]:
from seqeval.metrics import classification_report, f1_score

def evaluate_model(trainer, dataset, label_list):
    predictions, labels, _ = trainer.predict(dataset)
    preds = predictions.argmax(axis=2)

    true_preds = []
    true_labels = []

    for pred, lab in zip(preds, labels):
        p, l = [], []
        for pi, li in zip(pred, lab):
            if li != -100:
                p.append(label_list[pi])
                l.append(label_list[li])
        true_preds.append(p)
        true_labels.append(l)

    print(classification_report(true_labels, true_preds))
    print("F1 Score:", f1_score(true_labels, true_preds))

In [14]:
trainer_chunk = Trainer(
    model=model_chunk,
    args=args,
    train_dataset=CustomDataset(train_chunk_enc),
    eval_dataset=CustomDataset(val_chunk_enc)
)

trainer_pos = Trainer(
    model=model_pos,
    args=args,
    train_dataset=CustomDataset(train_pos_enc),
    eval_dataset=CustomDataset(val_pos_enc)
)

In [15]:
trainer_chunk.train()
trainer_pos.train()

Step,Training Loss
50,0.818363
100,0.308676
150,0.206686
200,0.160164
250,0.127506
300,0.106449
350,0.107316
400,0.086884
450,0.094764
500,0.121615


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4704, training_loss=0.09810391825870896, metrics={'train_runtime': 1469.0438, 'train_samples_per_second': 25.617, 'train_steps_per_second': 3.202, 'total_flos': 3380593239023616.0, 'train_loss': 0.09810391825870896, 'epoch': 3.0})

In [16]:
print("Chunking Results:")
evaluate_model(trainer_chunk, CustomDataset(val_chunk_enc), chunk_labels)

print("\nPOS Results:")
evaluate_model(trainer_pos, CustomDataset(val_pos_enc), pos_labels)

Chunking Results:


              precision    recall  f1-score   support

         LOC       0.96      0.97      0.97      1837
        MISC       0.87      0.89      0.88       922
         ORG       0.91      0.92      0.91      1341
         PER       0.98      0.98      0.98      1842

   micro avg       0.94      0.95      0.94      5942
   macro avg       0.93      0.94      0.93      5942
weighted avg       0.94      0.95      0.94      5942

F1 Score: 0.944621047348168

POS Results:


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171:

              precision    recall  f1-score   support

         ART       0.99      1.00      0.99       647
        CONJ       0.99      0.99      0.99      1163
          DJ       0.95      0.95      0.95      1757
          DP       0.98      0.98      0.98      1993
          DV       0.95      0.94      0.95      1112
         ERB       0.98      0.98      0.98      2655
          ET       0.99      1.00      0.99      1878
         NTJ       0.93      0.83      0.88       112
         OUN       0.94      0.94      0.94      3704
         RON       0.99      1.00      0.99      2153
        ROPN       0.89      0.89      0.89      1420
          UM       0.98      0.99      0.98       355
        UNCT       1.00      1.00      1.00      2945
          UX       0.99      1.00      0.99      1455
          YM       0.84      0.93      0.88        81
           _       0.05      0.04      0.05        23

   micro avg       0.97      0.97      0.97     23453
   macro avg       0.90   

In [28]:
def predict(sentence, model, tokenizer, id2label):
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    words = sentence.split()

    # Step 1: KEEP BatchEncoding (no return_tensors here)
    encoding = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        padding=True
    )

    # Step 2: Convert to tensors manually
    input_ids = torch.tensor([encoding["input_ids"]]).to(device)
    attention_mask = torch.tensor([encoding["attention_mask"]]).to(device)

    # Step 3: Forward pass
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    predictions = torch.argmax(outputs.logits, dim=2)[0]

    # Step 4: Now word_ids() works
    word_ids = encoding.word_ids()

    prev_word = None
    for idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        if word_id != prev_word:
            label = id2label[predictions[idx].item()]
            print(words[word_id], "->", label)
        prev_word = word_id

In [29]:
sentence = "John works at Google in California"

print("Chunking:")
predict(sentence, model_chunk, tokenizer, chunk_i2l)

print("\nPOS Tagging:")
predict(sentence, model_pos, tokenizer, pos_i2l)

Chunking:
John -> B-PER
works -> O
at -> O
Google -> B-ORG
in -> O
California -> B-LOC

POS Tagging:
John -> PROPN
works -> VERB
at -> ADP
Google -> PROPN
in -> ADP
California -> PROPN


Comparision

Task 7: Comparison

POS Tagging vs Chunking

POS Tagging assigns grammatical labels (like noun, verb, adjective) to each word.
→ It works at the word level and is relatively easy.
Chunking groups words into meaningful phrases (like noun phrases or verb phrases).
→ It works at the phrase level and is moderately complex.

In simple terms:

POS tells what each word is
Chunking tells how words are grouped

Report / Blog:
Differences between POS Tagging and Chunking:

POS tagging focuses on identifying the grammatical role of individual words, whereas chunking builds on POS tags to group words into phrases. POS operates at a lower level (word-level), while chunking provides higher-level structural understanding (phrase-level).

Challenges Faced:
Handling token alignment in transformer models (like BERT)
Errors due to incorrect preprocessing (vocab mismatch, tensor issues)
Difficulty in mapping subword tokens back to original words
Managing different label formats (POS vs chunk tags)

Observations and Insights:
POS tagging is simpler and acts as a foundation for chunking
Chunking provides better context by grouping related words
Transformer models improve accuracy but require careful preprocessing
Both tasks together give a deeper understanding of sentence structure